# 4. Q-Learning Fundamentals for Topology Optimization

## 4.1 Introduction to Q-Learning

Q-Learning is a model-free reinforcement learning algorithm that learns the optimal action-value function $Q^*(s,a)$ through direct interaction with the environment. Unlike dynamic programming methods, Q-Learning does not require knowledge of the transition model $P(s'|s,a)$, making it particularly suitable for topology optimization where the effects of design modifications are complex and difficult to model analytically.

### 4.1.1 Historical Development

Q-Learning was introduced by Chris Watkins in his 1989 PhD thesis and has since become one of the most widely used reinforcement learning algorithms:

- **1989**: Initial formulation by Chris Watkins
- **1992**: Convergence proof by Watkins and Dayan
- **2000s**: Extension to function approximation
- **2015**: Deep Q-Networks (DQN) breakthrough
- **2020s**: Application to engineering design problems

### 4.1.2 Why Q-Learning for Topology Optimization?

Q-Learning is particularly well-suited for topology optimization because:

- **Model-free**: No need to predict how design modifications affect performance
- **Off-policy**: Can learn optimal designs while exploring suboptimal strategies
- **Convergence guarantees**: Theoretical convergence to optimal Q-values
- **Sample efficiency**: Reuses experience through replay and learning

## 4.2 Q-Learning Algorithm Fundamentals

### 4.2.1 Core Update Rule

The Q-Learning update rule is the heart of the algorithm:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \left[ r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t) \right]$$

Where:
- $Q(s_t, a_t)$: Current estimate of action value
- $\alpha$: Learning rate ($0 < \alpha \leq 1$)
- $r_t$: Reward received at time step $t$
- $\gamma$: Discount factor ($0 \leq \gamma \leq 1$)
- $\max_{a'} Q(s_{t+1}, a')$: Maximum Q-value for next state

### 4.2.2 Temporal Difference Learning

Q-Learning is a temporal difference (TD) learning method that updates estimates based on the difference between predicted and actual outcomes:

$$\delta_t = r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)$$

The TD error $\delta_t$ represents the surprise or prediction error:
- **Positive TD error**: Outcome better than expected
- **Negative TD error**: Outcome worse than expected
- **Zero TD error**: Perfect prediction

### 4.2.3 Algorithm Pseudocode

```
Initialize Q(s,a) arbitrarily for all s ∈ S, a ∈ A
Initialize learning rate α, discount factor γ, exploration rate ε

for episode = 1 to N_episodes:
    Initialize state s_0
    
    for t = 0 to T_max:
        # Action selection (ε-greedy)
        if random() < ε:
            a_t = random_action()  # Exploration
        else:
            a_t = argmax_a Q(s_t, a)  # Exploitation
        
        # Execute action and observe result
        s_{t+1}, r_t = environment.step(a_t)
        
        # Q-value update
        Q(s_t, a_t) = Q(s_t, a_t) + α[r_t + γ max_a' Q(s_{t+1}, a') - Q(s_t, a_t)]
        
        s_t = s_{t+1}
        
        if s_t is terminal:
            break
```

## 4.3 Key Components and Hyperparameters

### 4.3.1 Learning Rate (α)

The learning rate controls how much new information overrides old information:

- **High α (0.7-1.0)**: Fast learning, but potentially unstable
- **Low α (0.1-0.3)**: Stable learning, but slow convergence
- **Adaptive α**: Variable learning rate that decreases over time

**Recommended for topology optimization**: $\alpha = 0.8$ (based on empirical testing)

### 4.3.2 Discount Factor (γ)

The discount factor determines the importance of future rewards:

- **γ ≈ 0**: Myopic agent, only considers immediate rewards
- **γ ≈ 1**: Far-sighted agent, values future rewards highly

For topology optimization:
- **γ = 0.95**: Balances immediate and long-term design improvements
- Enables consideration of design sequences that build toward optimal topologies

### 4.3.3 Exploration Strategy (ε-greedy)

Exploration is crucial for discovering novel topologies:

$$\pi(a|s) = \begin{cases} 1 - \varepsilon + \frac{\varepsilon}{|A|} & \text{if } a = \arg\max_{a'} Q(s,a') \\ \frac{\varepsilon}{|A|} & \text{otherwise} \end{cases}$$

**Exploration decay**:
$$\varepsilon_t = \max(\varepsilon_{\text{min}}, \varepsilon_{\text{max}} \cdot e^{-\lambda t})$$

Parameters for topology optimization:
- $\varepsilon_{\text{max}} = 1.0$: Start with full exploration
- $\varepsilon_{\text{min}} = 0.1$: Maintain minimum exploration
- $\lambda = 0.05$: Moderate decay rate

## 4.4 Q-Learning for Topology Optimization

### 4.4.1 State and Action Space Design

For topology optimization, we need to carefully design the state and action spaces:

**State Representation**:
```python
# Flatten 2D topology to 1D state vector
state = topology.flatten()  # Shape: (n*n,) for n×n grid
# Each element is 0 (void) or 1 (material)
```

**Action Space**:
```python
# Actions: Toggle material at grid position (i,j)
actions = [(i, j) for i in range(n) for j in range(n)]
# Total actions: n² for n×n grid
```

### 4.4.2 Reward Function for Topology Design

The reward function guides the optimization toward desirable topologies:

$$r(s_t, a_t, s_{t+1}) = w_1 \cdot \Delta \text{Performance} + w_2 \cdot \text{ConstraintPenalty} + w_3 \cdot \text{DesignBonus}$$

For SynRM design specifically:
$$r = w_T \cdot \Delta T + w_\eta \cdot \Delta \eta - w_{\text{ripple}} \cdot T_{\text{ripple}} - w_{\text{mass}} \cdot \Delta M$$

Where:
- $\Delta T$: Change in average torque
- $\Delta \eta$: Change in efficiency
- $T_{\text{ripple}}$: Torque ripple (penalty term)
- $\Delta M$: Change in mass (penalty if excessive)

### 4.4.3 Q-Table Structure

For discrete topology optimization, we maintain a Q-table:

**Q-Table Dimensions**: $|S| \times |A|$
- **$|S| = 2^{n^2}$**: All possible topologies for $n \times n$ grid
- **$|A| = n^2$**: Possible actions (toggle each cell)

**Memory Considerations**:
- For $6 \times 6$ grid: $2^{36} \times 36$ entries (infeasible)
        """Q-learning update function"""
- **Alternative**: Hierarchical state representation

## 4.5 Practical Implementation

### 4.5.1 Q-Table Initialization

```python
import numpy as np

class QLearningTopologyOptimizer:
    def __init__(self, grid_size, learning_rate=0.8, gamma=0.95):
        self.grid_size = grid_size
        self.n_states = 2**(grid_size * grid_size)
        self.n_actions = grid_size * grid_size
        
        # Initialize Q-table
        self.q_table = np.zeros((self.n_states, self.n_actions))
        
        # Hyperparameters
        self.alpha = learning_rate  # Learning rate
        self.gamma = gamma          # Discount factor
        self.epsilon = 1.0          # Exploration rate
```

### 4.5.2 State Encoding and Decoding

```python
def encode_state(self, topology):
    """Convert 2D topology to state index"""
    flattened = topology.flatten()
    state_index = 0
    for i, bit in enumerate(flattened):
        state_index += bit * (2**i)
    return state_index

def decode_state(self, state_index):
    """Convert state index back to 2D topology"""
    flattened = np.zeros(self.grid_size * self.grid_size)
    for i in range(len(flattened)):
        flattened[i] = (state_index >> i) & 1
    return flattened.reshape((self.grid_size, self.grid_size))
```

### 4.5.3 Action Selection and Execution

```python
def select_action(self, state_index, training=True):
    """ε-greedy action selection"""
    if training and np.random.random() < self.epsilon:
        # Exploration: random action
        return np.random.randint(self.n_actions)
    else:
        # Exploitation: best known action
        return np.argmax(self.q_table[state_index])

def execute_action(self, topology, action):
    """Execute action and return new topology"""
    new_topology = topology.copy()
    i, j = action // self.grid_size, action % self.grid_size
    new_topology[i, j] = 1 - new_topology[i, j]  # Toggle cell
    return new_topology
```

### 4.5.4 Q-Value Update

```python
def update_q_value(self, state_index, action, reward, next_state_index):
    """Update Q-value using TD learning"""
    current_q = self.q_table[state_index, action]
    max_next_q = np.max(self.q_table[next_state_index])
    
    # TD update rule
    new_q = current_q + self.alpha * (reward + self.gamma * max_next_q - current_q)
    self.q_table[state_index, action] = new_q
```

## 4.6 Training Process

### 4.6.1 Episode Structure

Each episode represents a complete design process:

1. **Initialization**: Start from a baseline topology
2. **Iteration**: Make sequential design modifications
3. **Termination**: Stop when convergence or max steps reached
4. **Evaluation**: Record final design performance

### 4.6.2 Training Loop

```python
def train(self, n_episodes, max_steps_per_episode):
    """Main training loop"""
    rewards_history = []
    best_designs = []
    
    for episode in range(n_episodes):
        # Initialize episode
        topology = self.initialize_topology()
        episode_reward = 0
        
        for step in range(max_steps_per_episode):
            # Convert topology to state index
            state_index = self.encode_state(topology)
            
            # Select and execute action
            action = self.select_action(state_index)
            new_topology = self.execute_action(topology, action)
            
            # Calculate reward
            reward = self.calculate_reward(topology, new_topology)
            episode_reward += reward
            
            # Update Q-value
            next_state_index = self.encode_state(new_topology)
            self.update_q_value(state_index, action, reward, next_state_index)
            
            topology = new_topology
            
        # Decay exploration rate
        self.epsilon = max(self.epsilon_min, 
                          self.epsilon * self.epsilon_decay)
        
        # Record results
        rewards_history.append(episode_reward)
        if episode % 10 == 0:
            best_designs.append(topology.copy())
    
    return rewards_history, best_designs
```

## 4.7 Convergence and Performance

### 4.7.1 Convergence Criteria

Q-Learning converges under specific conditions:

**Robbins-Monro Conditions**:
$$\sum_{t=0}^{\infty} \alpha_t = \infty$$
$$\sum_{t=0}^{\infty} \alpha_t^2 < \infty$$

**Practical Implementation**:
- **Learning rate decay**: $\alpha_t = \alpha_0 / (1 + \beta t)$
- **Exploration decay**: Gradually reduce $\varepsilon$
- **Visit frequency**: All state-action pairs visited infinitely often

### 4.7.2 Performance Metrics

**Learning Metrics**:
- **Episode rewards**: Track learning progress
- **Q-value convergence**: Stability of learned values
- **Action consistency**: Convergence to stable policy

**Design Metrics**:
- **Final performance**: Torque, efficiency, etc.
- **Convergence speed**: Episodes to reach good designs
- **Design diversity**: Variety of discovered topologies

### 4.7.3 Common Issues and Solutions

**Slow Convergence**:
- **Problem**: Large state space leads to slow learning
- **Solution**: Function approximation, curriculum learning

**Local Optima**:
- **Problem**: Agent gets stuck in suboptimal designs
- **Solution**: Higher exploration rate, reward shaping

**Sparse Rewards**:
- **Problem**: Long delay between actions and meaningful feedback
- **Solution**: Intermediate rewards, reward shaping

## 4.8 Advanced Q-Learning Variants

### 4.8.1 Deep Q-Networks (DQN)

For large-scale topology optimization:

$$Q(s,a; \theta) \approx Q^*(s,a)$$

Where $\theta$ are neural network parameters.

**Key Innovations**:
- **Experience replay**: Store and reuse transitions
- **Target networks**: Stable learning targets
- **CNN architectures**: Spatial feature extraction

### 4.8.2 Double Q-Learning

Addresses overestimation bias:

$$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma Q'(s', \arg\max_a Q(s',a)) - Q(s,a)]$$

Where $Q'$ is the target network.

### 4.8.3 Multi-step Q-Learning

Uses n-step returns for faster learning:

$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n \max_a Q(s_{t+n}, a)$$

## 4.9 Summary

Q-Learning provides a powerful framework for topology optimization:

### 4.9.1 Key Advantages

- **Model-free**: No need for analytical design models
- **Off-policy**: Can learn optimal designs while exploring
- **Theoretical guarantees**: Convergence to optimal solution
- **Adaptive**: Learns from experience and improves over time

### 4.9.2 Practical Considerations

- **State space size**: Exponential growth requires approximation
- **Reward design**: Critical for guiding optimization
- **Exploration balance**: Essential for discovering novel designs
- **Computational cost**: Many iterations required for convergence

Q-Learning represents a fundamental shift from traditional optimization methods, enabling automated discovery of innovative electrical machine topologies through sequential learning and exploration.

---

**Next Section**: [SynRM Environment](05_synrm_environment.ipynb) - Application domain specifics and electromagnetic design fundamentals.

```{figure} ../_static/figures/Tree_structure.png---name: fig-tree-search-structurewidth: 80%---Tree search structure for sequential decision-making in topology optimization. Each node represents a topology state, branches represent possible actions (material modifications), and the search explores design space systematically to find optimal configurations through Q-value updates.```

In [ ]:
# Q-Learning demonstration for topology optimization
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, Markdown
import seaborn as sns

class SimpleQLearningTO:
    """Simplified Q-Learning for Topology Optimization"""
    
    def __init__(self, grid_size=4, learning_rate=0.8, gamma=0.95, epsilon=1.0):
        self.grid_size = grid_size
        self.n_cells = grid_size * grid_size
        self.n_actions = self.n_cells  # Toggle each cell
        
        # Hyperparameters
        self.alpha = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.1
        
        # Initialize Q-table (simplified: use smaller state representation)
        # For demonstration, we'll use a compressed state representation
        self.q_table = np.random.randn(16, self.n_actions) * 0.1  # 16 compressed states
        
        # Training history
        self.reward_history = []
        self.topology_history = []
    
    def compress_state(self, topology):
        """Compress topology to smaller state space for demo"""
        # Simple compression: count material in each quadrant
        half = self.grid_size // 2
        state_vector = []
        
        for i in range(0, self.grid_size, half):
            for j in range(0, self.grid_size, half):
                quadrant = topology[i:i+half, j:j+half]
                count = np.sum(quadrant)
                state_vector.append(min(count, 3))  # Cap at 3 for 4 states per quadrant
        
        # Convert to single state index
        state_index = 0
        for i, val in enumerate(state_vector):
            state_index += val * (4**i)
        
        return state_index
    
    def select_action(self, state_index, training=True):
        """ε-greedy action selection"""
        if training and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.q_table[state_index])
    
    def execute_action(self, topology, action):
        """Execute action (toggle cell)"""
        new_topology = topology.copy()
        i, j = action // self.grid_size, action % self.grid_size
        new_topology[i, j] = 1 - new_topology[i, j]
        return new_topology
    
    def calculate_reward(self, old_topology, new_topology):
        """Calculate reward based on design improvement"""
        # Reward function for demonstration
        
        # Connectivity reward
        old_connectivity = self.measure_connectivity(old_topology)
        new_connectivity = self.measure_connectivity(new_topology)
        connectivity_reward = (new_connectivity - old_connectivity) * 10
        
        # Material usage penalty
        material_ratio = np.sum(new_topology) / self.n_cells
        material_penalty = -abs(material_ratio - 0.5) * 5
        
        # Symmetry bonus
        symmetry_bonus = self.measure_symmetry(new_topology) * 2
        
        return connectivity_reward + material_penalty + symmetry_bonus
    
    def measure_connectivity(self, topology):
        """Simple connectivity measure"""
        connectivity = 0
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if topology[i, j] == 1:
                    for di, dj in [(0,1), (1,0)]:
                        ni, nj = i + di, j + dj
                        if 0 <= ni < self.grid_size and 0 <= nj < self.grid_size:
                            if topology[ni, nj] == 1:
                                connectivity += 0.5
        return connectivity
    
    def measure_symmetry(self, topology):
        """Measure symmetry (0 to 1)"""
        h_sym = 1 - np.mean(np.abs(topology - np.fliplr(topology)))
        v_sym = 1 - np.mean(np.abs(topology - np.flipud(topology)))
        return (h_sym + v_sym) / 2
    
    def update_q_value(self, state_index, action, reward, next_state_index):
        """Q-learning update rule"""
        current_q = self.q_table[state_index, action]
        max_next_q = np.max(self.q_table[next_state_index])
        
        # TD update
        new_q = current_q + self.alpha * (reward + self.gamma * max_next_q - current_q)
        self.q_table[state_index, action] = new_q
    
    def train_episode(self, max_steps=20):
        """Train for one episode"""
        # Initialize with some random topology
        topology = np.random.choice([0, 1], size=(self.grid_size, self.grid_size), p=[0.7, 0.3])
        episode_reward = 0
        episode_topologies = [topology.copy()]
        
        for step in range(max_steps):
            state_index = self.compress_state(topology)
            action = self.select_action(state_index)
            new_topology = self.execute_action(topology, action)
            
            reward = self.calculate_reward(topology, new_topology)
            episode_reward += reward
            
            next_state_index = self.compress_state(new_topology)
            self.update_q_value(state_index, action, reward, next_state_index)
            
            topology = new_topology
            episode_topologies.append(topology.copy())
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return episode_reward, episode_topologies

# Demonstrate Q-Learning
agent = SimpleQLearningTO(grid_size=4)

# Train for multiple episodes
n_episodes = 50
training_rewards = []
best_episodes = []

print("Training Q-Learning Agent for Topology Optimization...")
for episode in range(n_episodes):
    reward, topologies = agent.train_episode(max_steps=15)
    training_rewards.append(reward)
    
    # Store best episodes for visualization
    if episode % 10 == 0:
        best_episodes.append((episode, topologies[-1], reward))
    
    if (episode + 1) % 10 == 0:
        print(f"Episode {episode + 1}: Reward = {reward:.2f}, Epsilon = {agent.epsilon:.3f}")

# Visualize training progress
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Reward progression
ax1 = axes[0, 0]
ax1.plot(training_rewards, 'b-', alpha=0.7)
ax1.plot(range(len(training_rewards)), np.maximum.accumulate(training_rewards), 'r-', linewidth=2)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Episode Reward')
ax1.set_title('Q-Learning Training Progress')
ax1.legend(['Episode Reward', 'Best So Far'])
ax1.grid(True, alpha=0.3)

# Plot 2: Epsilon decay
ax2 = axes[0, 1]
epsilons = [agent.epsilon * (agent.epsilon_decay ** i) for i in range(n_episodes)]
epsilons = [max(agent.epsilon_min, eps) for eps in epsilons]
ax2.plot(epsilons, 'g-')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Exploration Rate (ε)')
ax2.set_title('Exploration-Exploitation Balance')
ax2.grid(True, alpha=0.3)

# Plot 3: Final topology
ax3 = axes[1, 0]
final_topology = best_episodes[-1][1]
im = ax3.imshow(final_topology, cmap='binary', interpolation='nearest')
ax3.set_title(f'Final Design\nReward: {best_episodes[-1][2]:.2f}')
ax3.set_xlabel('X Position')
ax3.set_ylabel('Y Position')
ax3.grid(True, alpha=0.3)

# Plot 4: Q-value heatmap (sample)
ax4 = axes[1, 1]
sample_q = agent.q_table[:min(16, len(agent.q_table)), :min(16, agent.n_actions)]
sns.heatmap(sample_q, ax=ax4, cmap='viridis', cbar_kws={'label': 'Q-value'})
ax4.set_title('Q-Table Sample (State × Action)')
ax4.set_xlabel('Action')
ax4.set_ylabel('State')

plt.suptitle('Q-Learning for Topology Optimization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

display(MMarkdown("""
**Figure: Q-Learning Demonstration Results**

This demonstration shows Q-Learning applied to topology optimization:

1. **Training Progress**: The agent learns to improve design quality over episodes, with the red line showing the best reward achieved so far.

2. **Exploration Decay**: The exploration rate ε starts high (exploring many designs) and gradually decreases (exploiting known good designs).

3. **Final Design**: The learned topology shows balanced material distribution with good connectivity and symmetry.

4. **Q-Table**: The heatmap shows learned Q-values for different state-action pairs, with brighter colors indicating higher expected rewards.

**Key Insights from Q-Learning:**
- The agent learns which design modifications lead to better performance
- Exploration is crucial for discovering novel topologies
- The balance between exploration and exploitation changes over time
- Q-values encode the long-term value of design decisions

This simplified demonstration captures the essence of how Q-Learning can be applied to real topology optimization problems in electrical machine design.
"""))